In [ ]:
import json
import numpy as np
from google.colab import drive
from collections import Counter

drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive'

def load_labels(path):
    with open(path) as f:
        return json.load(f)

medqa_traps = load_labels(f'{PROJECT_DIR}/bias_labels.json')
medqa_nontraps = load_labels(f'{PROJECT_DIR}/non_trap_bias_labels.json')
medqa_shared = load_labels(f'{PROJECT_DIR}/shared_failure_bias_labels.json')

medmcqa_unanim = load_labels(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_labels.json')

medqa_gpt_labels = load_labels(f'{PROJECT_DIR}/bias_labels_gpt4omini.json')
medmcqa_gpt_labels = load_labels(f'{PROJECT_DIR}/medmcqa_unanimous_wrong_bias_gpt4omini.json')

from datasets import load_dataset
medqa = load_dataset("GBaker/MedQA-USMLE-4-options")
medqa_test = medqa['test']

with open(f'{PROJECT_DIR}/medmcqa_sample.json') as f:
    medmcqa_questions = {q['id']: q for q in json.load(f)}

with open(f'{PROJECT_DIR}/llama_results.json') as f:
    medqa_llama = {r['idx']: r for r in json.load(f)}

print(f"MedQA traps labeled: {len(medqa_traps)}")
print(f"MedQA non-traps labeled: {len(medqa_nontraps)}")
print(f"MedQA shared-failure labeled: {len(medqa_shared)}")
print(f"MedMCQA unanimous-wrong labeled: {len(medmcqa_unanim)}")
print(f"\nMedQA test questions loaded: {len(medqa_test)}")
print(f"MedMCQA questions loaded: {len(medmcqa_questions)}")

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'sentence-transformers'], check=True)

from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading sentence-transformers model...")
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"Model loaded\n")

def split_into_thirds(text):
    words = text.split()
    n = len(words)
    if n < 9:
        return None
    third = n // 3
    return {
        'early': ' '.join(words[:third]),
        'middle': ' '.join(words[third:2*third]),
        'late': ' '.join(words[2*third:]),
    }

def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def pc_structure_score(question_text, wrong_answer_text):
    thirds = split_into_thirds(question_text)
    if thirds is None:
        return None
    wrong_emb = encoder.encode(wrong_answer_text, show_progress_bar=False)
    early_emb = encoder.encode(thirds['early'], show_progress_bar=False)
    middle_emb = encoder.encode(thirds['middle'], show_progress_bar=False)
    late_emb = encoder.encode(thirds['late'], show_progress_bar=False)
    sim_early = cosine_sim(wrong_emb, early_emb)
    sim_middle = cosine_sim(wrong_emb, middle_emb)
    sim_late = cosine_sim(wrong_emb, late_emb)
    return sim_early, sim_middle, sim_late, sim_early - sim_late

rows = []
skipped_none = 0

for b in medqa_traps:
    idx = b['idx']
    wrong_letter = medqa_llama[idx].get('pred')
    if wrong_letter is None or wrong_letter not in 'ABCD':
        skipped_none += 1
        continue
    q = medqa_test[idx]
    rows.append({
        'dataset': 'medqa', 'subset': 'trap', 'id': idx,
        'bias_label': b['bias_type'],
        'question': q['question'],
        'wrong_answer': q['options'][wrong_letter],
    })

for b in medqa_nontraps:
    if b['bias_type'] == 'ERROR':
        skipped_none += 1
        continue
    idx = b['idx']
    wrong_letter = medqa_llama[idx].get('pred')
    if wrong_letter is None or wrong_letter not in 'ABCD':
        skipped_none += 1
        continue
    q = medqa_test[idx]
    rows.append({
        'dataset': 'medqa', 'subset': 'nontrap', 'id': idx,
        'bias_label': b['bias_type'],
        'question': q['question'],
        'wrong_answer': q['options'][wrong_letter],
    })

for b in medmcqa_unanim:
    if b.get('bias_type') == 'ERROR' or b.get('modal_wrong') not in 'ABCD':
        skipped_none += 1
        continue
    qid = b['id']
    q = medmcqa_questions[qid]
    rows.append({
        'dataset': 'medmcqa', 'subset': 'unanimous_wrong', 'id': qid,
        'bias_label': b['bias_type'],
        'question': q['question'],
        'wrong_answer': q['options'][b['modal_wrong']],
    })

print(f"Total labeled rows to score: {len(rows)}")
print(f"Skipped (None pred or ERROR label): {skipped_none}")
print(f"  MedQA traps:               {sum(1 for r in rows if r['subset']=='trap')}")
print(f"  MedQA non-traps:           {sum(1 for r in rows if r['subset']=='nontrap')}")
print(f"  MedMCQA unanimous-wrong:   {sum(1 for r in rows if r['subset']=='unanimous_wrong')}")

print(f"\nComputing PC-structure scores...")
skipped_short = 0
for i, row in enumerate(rows):
    result = pc_structure_score(row['question'], row['wrong_answer'])
    if result is None:
        row['pc_score'] = None
        skipped_short += 1
    else:
        sim_e, sim_m, sim_l, pc = result
        row['sim_early'] = sim_e
        row['sim_middle'] = sim_m
        row['sim_late'] = sim_l
        row['pc_score'] = pc
    if (i+1) % 100 == 0:
        print(f"  {i+1}/{len(rows)}")

print(f"\nSkipped (question too short): {skipped_short}")
print(f"Scored: {len(rows) - skipped_short}")

with open(f'{PROJECT_DIR}/programmatic_pc_scores.json', 'w') as f:
    json.dump(rows, f, indent=2)
print(f"Saved programmatic_pc_scores.json")

scored = [r for r in rows if r['pc_score'] is not None]
pc_scores = [r['pc_score'] for r in scored]
print(f"\n=== PC-structure score distribution (all rows) ===")
print(f"  N: {len(scored)}")
print(f"  Mean: {np.mean(pc_scores):+.4f}")
print(f"  Std:  {np.std(pc_scores):.4f}")
print(f"  Min:  {np.min(pc_scores):+.4f}")
print(f"  Max:  {np.max(pc_scores):+.4f}")
print(f"  % positive (wrong closer to early): {sum(1 for s in pc_scores if s > 0)/len(pc_scores)*100:.1f}%")

In [ ]:
from scipy import stats
import numpy as np

scored = [r for r in rows if r['pc_score'] is not None]

pc_labeled = [r for r in scored if r['bias_label'] == 'PREMATURE_CLOSURE']
anchor_labeled = [r for r in scored if r['bias_label'] == 'ANCHORING_BIAS']
other_labeled = [r for r in scored if r['bias_label'] not in ('PREMATURE_CLOSURE', 'ANCHORING_BIAS', 'ERROR')]

print(f"=== PC-score by LLM-classifier label (Llama-3.3-70B) ===\n")
print(f"{'Label':<22} {'N':<6} {'Mean PC-score':<15} {'Std':<8} {'% positive':<12}")
print("-" * 65)
for label, group in [('PREMATURE_CLOSURE', pc_labeled),
                     ('ANCHORING_BIAS', anchor_labeled),
                     ('AVAILABILITY+OTHER', other_labeled)]:
    if not group:
        continue
    scores = [r['pc_score'] for r in group]
    pos_pct = sum(1 for s in scores if s > 0) / len(scores) * 100
    print(f"{label:<22} {len(group):<6} {np.mean(scores):+.4f}          {np.std(scores):.4f}   {pos_pct:.1f}%")

if len(pc_labeled) > 10 and len(anchor_labeled) > 10:
    pc_scores = [r['pc_score'] for r in pc_labeled]
    an_scores = [r['pc_score'] for r in anchor_labeled]
    t, p = stats.ttest_ind(pc_scores, an_scores, equal_var=False)
    print(f"\nWelch's t-test (PC vs Anchoring PC-scores):")
    print(f"  t = {t:.3f}")
    print(f"  p = {p:.4f}")
    if p < 0.05 and t > 0:
        print(f"  → PC-labeled questions score significantly HIGHER (validated direction)")
    elif p < 0.05 and t < 0:
        print(f"  → PC-labeled questions score significantly LOWER (opposite direction)")
    else:
        print(f"  → No significant difference (rule doesn't discriminate)")

print(f"\n=== Per-dataset breakdown ===\n")
for ds_subset in [('medqa', 'trap'), ('medqa', 'nontrap'), ('medmcqa', 'unanimous_wrong')]:
    group = [r for r in scored if r['dataset'] == ds_subset[0] and r['subset'] == ds_subset[1]]
    if not group:
        continue
    pc_g = [r['pc_score'] for r in group if r['bias_label'] == 'PREMATURE_CLOSURE']
    an_g = [r['pc_score'] for r in group if r['bias_label'] == 'ANCHORING_BIAS']
    print(f"{ds_subset[0]:<10} {ds_subset[1]:<20}")
    if pc_g:
        print(f"  PC-labeled        (n={len(pc_g):3d}): mean={np.mean(pc_g):+.4f}")
    if an_g:
        print(f"  Anchoring-labeled (n={len(an_g):3d}): mean={np.mean(an_g):+.4f}")
    print()